# Mumbai Festival Noise Risk PredictorThis notebook takes a location (lat/lon or address), matches it to the nearest MPCB monitoringpoints, predicts festival-noise levels using a small ML model trained on real Mumbai Diwali data,spreads that into a 24-hour curve using real Ganesh Chaturthi hourly data, and translates it intoa risk level for pets, children, the elderly, and pregnant people.**Data used:**- `mumbai_diwali_2023_2024.csv` — 15 Mumbai locations, day/night average dB, pre/main/post-Diwali- `mumbai_ganesh_2024_hourly.csv` — 25 Mumbai locations, real hourly dB from 18:00–24:00 (immersion window)- Jabalpur Navratri file is **excluded** — different city, not used**Important caveat:** with ~140 total rows and only 2 known Mumbai events, this is a workingprototype, not a calibrated forecasting tool. Treat the output as an estimate, not a live reading.

## Step 0 — Setup & upload files

In [ ]:
!pip install ipywidgets --quietimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport ipywidgets as widgetsfrom IPython.display import display, clear_output# --- Upload the 3 CSVs (Colab file picker). If running locally, just place them# in the working directory and skip this cell. ---try:    from google.colab import files    print("Upload mumbai_diwali_2023_2024.csv and mumbai_ganesh_2024_hourly.csv")    uploaded = files.upload()except ImportError:    print("Not running in Colab — expecting CSVs already in the working directory.")

## Step 1 — Load & clean data

In [ ]:
diwali = pd.read_csv('mumbai_diwali_2023_2024.csv')ganesh = pd.read_csv('mumbai_ganesh_2024_hourly.csv')print("Diwali:", diwali.shape, "| locations:", diwali.location.nunique())print("Ganesh:", ganesh.shape, "| locations:", ganesh.location.nunique())diwali.head()

## Step 2 — Master location table (lat/lon + zone type)MPCB doesn't publish coordinates in these exports, so this is a manually-built lookup for the~38 unique location names across both datasets. **These coordinates are approximate** (city-levelknowledge, not surveyed) — replace with real geocoding (Google Maps API / Nominatim) beforerelying on this for anything beyond a prototype.

In [ ]:
LOCATION_INFO = {    'Colaba': (18.9067, 72.8147, 'Silence'),    'Mantralaya': (18.9257, 72.8235, 'Silence'),    'Mazgaon': (18.9647, 72.8479, 'Commercial'),    'Girgaon': (18.9515, 72.8198, 'Commercial'),    'Hindu Colony': (19.0176, 72.8479, 'Residential'),    'Kamathipura': (18.9647, 72.8258, 'Commercial'),    'Mahim': (19.0410, 72.8397, 'Residential'),    'Malabar Hills': (18.9548, 72.7960, 'Residential'),    'Matunga': (19.0272, 72.8547, 'Residential'),    'Byculla': (18.9750, 72.8330, 'Commercial'),    'Dadar': (19.0178, 72.8478, 'Commercial'),    'Parel': (19.0080, 72.8410, 'Commercial'),    'Prabhadevi': (19.0170, 72.8250, 'Residential'),    'Sion': (19.0430, 72.8620, 'Residential'),    'Worli': (19.0100, 72.8170, 'Residential'),    'Andheri': (19.1197, 72.8468, 'Commercial'),    'Bandra': (19.0596, 72.8295, 'Commercial'),    'Bhandup': (19.1440, 72.9370, 'Residential'),    'Borivali': (19.2288, 72.8567, 'Residential'),    'Chembur (East)': (19.0553, 72.9042, 'Residential'),    'Chembur (West)': (19.0500, 72.8950, 'Residential'),    'Chinchpokali (E)': (18.9800, 72.8340, 'Commercial'),    'Chinchpokali (W)': (18.9790, 72.8300, 'Commercial'),    'Dadar (East)': (19.0189, 72.8508, 'Commercial'),    'Dadar (West)': (19.0170, 72.8420, 'Commercial'),    'Elphinstone': (19.0000, 72.8300, 'Commercial'),    'Ghatkopar (E)': (19.0860, 72.9081, 'Residential'),    'Girgaon Chowpati': (18.9530, 72.8138, 'Residential'),    'Grant Road': (18.9630, 72.8145, 'Commercial'),    'Juhu Chowpati': (19.1075, 72.8263, 'Residential'),    'Kandivali (East)': (19.2039, 72.8697, 'Residential'),    'Kandivali (West)': (19.2065, 72.8360, 'Residential'),    'Khar': (19.0728, 72.8370, 'Residential'),    'Mulund': (19.1726, 72.9425, 'Residential'),    'Mumbai Central': (18.9700, 72.8194, 'Commercial'),    'Santacruz (East)': (19.0825, 72.8562, 'Residential'),    'Vikhroli': (19.1090, 72.9280, 'Residential'),    'Wadala': (19.0170, 72.8610, 'Residential'),}loc_df = pd.DataFrame([    {'location': k, 'lat': v[0], 'lon': v[1], 'zone_type': v[2]}    for k, v in LOCATION_INFO.items()])missing = (set(diwali.location.unique()) | set(ganesh.location.unique())) - set(LOCATION_INFO)assert not missing, f"Add coordinates for: {missing}"print(f"{len(loc_df)} locations mapped, zone types: {loc_df.zone_type.value_counts().to_dict()}")loc_df.head()

## Step 3 — Hourly shape curveThe Ganesh dataset is the only one with true hourly readings, and only for 18:00–24:00(the immersion / procession window). We derive real dB offsets for those 6 hours, and usegeneric, commonly-observed urban diurnal noise patterns for the remaining 18 hours since noground truth exists for them in this data — **that part is an assumption, not measured**.

In [ ]:
hcols = ['h18_19','h19_20','h20_21','h21_22','h22_23','h23_24']ganesh_hourly_mean = ganesh[hcols].mean()evening_offset = ganesh_hourly_mean - ganesh_hourly_mean.mean()  # dB deviation from evening mean# Generic offsets for hours with no ground truth (assumption, not measured):# night trough ~2-4am, gentle daytime rise into eveningHOURLY_OFFSET_DB = {    0: -8, 1: -10, 2: -11, 3: -11, 4: -9, 5: -6,    6: -2, 7: 1, 8: 2, 9: 2, 10: 1, 11: 1, 12: 1, 13: 1, 14: 0, 15: 0, 16: 0, 17: 1,}for i, h in enumerate(range(18, 24)):    HOURLY_OFFSET_DB[h] = round(float(evening_offset.iloc[i]), 2)pd.Series(HOURLY_OFFSET_DB, name='dB offset from period average').sort_index()

## Step 4 — Train the prediction modelTrained on the Diwali dataset (the only one with `zone_type` + `day_type` + true day/night Leq).With ~90 rows this is intentionally simple — a Random Forest predicting `day_leq` and `night_leq`from zone type and festival-day stage. Leave-one-out MAE is reported for a rough sense of errormargin (expect several dB of error given the data size).

In [ ]:
from sklearn.ensemble import RandomForestRegressorfrom sklearn.preprocessing import OneHotEncoderfrom sklearn.compose import ColumnTransformerfrom sklearn.pipeline import Pipelinefrom sklearn.model_selection import LeaveOneOutfrom sklearn.metrics import mean_absolute_errortrain = diwali.merge(loc_df[['location','zone_type']], on='location', suffixes=('','_lut'))X = train[['zone_type', 'day_type']]y = train[['day_leq', 'night_leq']]pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['zone_type','day_type'])])model = Pipeline([('pre', pre), ('rf', RandomForestRegressor(n_estimators=200, random_state=42))])loo = LeaveOneOut()preds, actuals = [], []for tr_idx, te_idx in loo.split(X):    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])    preds.append(model.predict(X.iloc[te_idx])[0])    actuals.append(y.iloc[te_idx].values[0])preds, actuals = np.array(preds), np.array(actuals)print(f"Leave-one-out MAE — day_leq: {mean_absolute_error(actuals[:,0], preds[:,0]):.1f} dB, "      f"night_leq: {mean_absolute_error(actuals[:,1], preds[:,1]):.1f} dB")print("(a few dB of error is expected with this little data — treat outputs as estimates)")model.fit(X, y)  # final model on all available data

## Step 5 — Nearest-match + risk thresholdsFinds the closest known monitoring point(s) to any lat/lon, and defines simple dB thresholdsper vulnerable group (illustrative, loosely WHO/CPCB-style — adjust to your own sourcing).

In [ ]:
def haversine(lat1, lon1, lat2, lon2):    R = 6371    p1, p2 = np.radians(lat1), np.radians(lat2)    dphi = np.radians(lat2 - lat1)    dlmb = np.radians(lon2 - lon1)    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlmb/2)**2    return 2 * R * np.arcsin(np.sqrt(a))def nearest_locations(lat, lon, k=3):    d = loc_df.copy()    d['dist_km'] = haversine(lat, lon, d.lat, d.lon)    return d.sort_values('dist_km').head(k)THRESHOLDS = {    'general':  {'low': 55, 'medium': 70},    'child':    {'low': 50, 'medium': 65},    'elderly':  {'low': 50, 'medium': 63},    'pregnant': {'low': 50, 'medium': 63},    'pet':      {'low': 45, 'medium': 60},}def risk_level(db, group):    t = THRESHOLDS[group]    if db < t['low']: return 'Low'    if db < t['medium']: return 'Medium'    return 'High'

## Step 6 — Put it all together: predict for any location

In [ ]:
def predict_noise_risk(lat, lon, day_type='diwali', k_nearest=2):    """    lat, lon   : coordinates of the point to predict for    day_type   : 'pre_diwali' | 'diwali' | 'post_diwali'  (matches model's training labels)    Returns a DataFrame: hour, predicted_dB, risk_general, risk_child, risk_elderly, risk_pregnant, risk_pet    plus the matched location(s) used.    """    near = nearest_locations(lat, lon, k=k_nearest)    zone = near.iloc[0].zone_type  # dominant nearest zone type    X_pred = pd.DataFrame([{'zone_type': zone, 'day_type': day_type}])    day_leq, night_leq = model.predict(X_pred)[0]    rows = []    for h in range(24):        base = day_leq if 7 <= h < 21 else night_leq  # rough day/night window        db = round(base + HOURLY_OFFSET_DB[h], 1)        rows.append({            'hour': h,            'predicted_dB': db,            'risk_general': risk_level(db, 'general'),            'risk_child': risk_level(db, 'child'),            'risk_elderly': risk_level(db, 'elderly'),            'risk_pregnant': risk_level(db, 'pregnant'),            'risk_pet': risk_level(db, 'pet'),        })    return pd.DataFrame(rows), nearresult, matched = predict_noise_risk(19.02, 72.84, day_type='diwali')print("Matched to:", matched[['location','dist_km','zone_type']].to_string(index=False))result

## Step 7 — Interactive UI (enter a location, see the 24h risk chart)

In [ ]:
lat_box = widgets.FloatText(value=19.02, description='Latitude:')lon_box = widgets.FloatText(value=72.84, description='Longitude:')day_dd  = widgets.Dropdown(options=['pre_diwali','diwali','post_diwali'], value='diwali', description='Day:')group_dd = widgets.Dropdown(options=['general','child','elderly','pregnant','pet'], value='general', description='Group:')run_btn = widgets.Button(description='Predict', button_style='primary')out = widgets.Output()RISK_COLOR = {'Low': '#4caf50', 'Medium': '#ff9800', 'High': '#e53935'}def on_click(b):    with out:        clear_output()        result, matched = predict_noise_risk(lat_box.value, lon_box.value, day_dd.value)        print("Nearest monitored location(s):")        print(matched[['location','dist_km','zone_type']].to_string(index=False))        colors = [RISK_COLOR[r] for r in result[f'risk_{group_dd.value}']]        fig, ax = plt.subplots(figsize=(10,4))        ax.bar(result.hour, result.predicted_dB, color=colors)        ax.set_xlabel('Hour of day'); ax.set_ylabel('Predicted dB (Leq)')        ax.set_title(f'Predicted 24h noise & {group_dd.value} risk — {day_dd.value}')        ax.set_xticks(range(0,24,2))        plt.tight_layout(); plt.show()run_btn.on_click(on_click)display(widgets.VBox([lat_box, lon_box, day_dd, group_dd, run_btn, out]))

## Step 8 — Sanity check against real dataFor each known Diwali location, hide its real value, predict using only zone_type + day_type,and compare. With this little data expect noticeable error — this is about seeing *how rough*the estimate is, not proving accuracy.

In [ ]:
check = train.copy()check['pred_day_leq'] = preds[:,0] if len(preds)==len(check) else np.nan# recompute LOO predictions aligned to rows for displayloo_preds = []for tr_idx, te_idx in LeaveOneOut().split(X):    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])    loo_preds.append(model.predict(X.iloc[te_idx])[0])loo_preds = np.array(loo_preds)check['pred_day_leq'] = loo_preds[:,0]check['pred_night_leq'] = loo_preds[:,1]check['day_error'] = (check.pred_day_leq - check.day_leq).round(1)check['night_error'] = (check.pred_night_leq - check.night_leq).round(1)model.fit(X, y)  # restore final modelcheck[['location','day_type','day_leq','pred_day_leq','day_error','night_leq','pred_night_leq','night_error']]